# 00 · Configuração do pipeline de modelos desafiantes de PD e LGD

Este notebook **não processa dados**. Ele define três coisas que todos os outros notebooks usam:

1. **As tabelas de referência**: métricas de avaliação, técnicas de aprendizado de máquina e o *padrão de referência* (hiperparâmetros equivalentes entre técnicas) para PD e para LGD.
2. **O dicionário de configuração** (`cfg`): um único objeto com todas as escolhas de uma execução. Cada etapa lê dele os seus parâmetros, e ele é gravado junto com os resultados para que a execução possa ser reproduzida.
3. **Utilitários de saída**: gravação em tabelas Delta, registro no MLflow e exibição de tabelas.

**Como usar.** No topo de cada notebook do pipeline:

```python
%run ./00_config
cfg = cria_configuracao(tarefa="pd", metrica="ks", data_corte="2025-09-01",
                        esquema_saida="catalogo.esquema", experimento_mlflow="/Users/.../desafiante_pd")
```

**Glossário rápido**

| termo | significado |
|---|---|
| PD | probabilidade de inadimplência (*probability of default*): chance de o contrato entrar em default num horizonte, aqui 12 meses |
| LGD | perda dada a inadimplência (*loss given default*): fração da exposição que não é recuperada depois do default |
| safra | mês de concessão do contrato (na LGD, mês do default) |
| DES | amostra de desenvolvimento: safras usadas para construir o modelo; dividida em **treino** (primeiras safras) e **teste** (últimas safras) |
| OOT | amostra fora do tempo (*out of time*): safras posteriores ao DES, usadas para escolher a técnica vencedora e testar a régua |
| HIST | safras anteriores ao DES, usadas só para testar os horizontes longos da PD lifetime |
| régua | tabela que transforma o score em ratings (faixas), cada um com uma PD (ou LGD) fixa |

In [ ]:
import copy
import json
import uuid
from datetime import date, datetime
from importlib import metadata

import matplotlib.pyplot as plt

# As etapas devolvem os gráficos como objetos (para exibir, salvar e registrar
# no MLflow), então muitas figuras ficam abertas ao mesmo tempo, de propósito:
# a régua gera um gráfico por rating e o lifetime, cinco por rating. O aviso
# padrão do matplotlib ("mais de 20 figuras abertas") é desligado.
plt.rcParams["figure.max_open_warning"] = 0

## 1. Métricas, técnicas e padrão de referência

Três tabelas fixas, por tarefa. Elas definem **o que** pode ser escolhido na configuração; a configuração define **o que foi** escolhido numa execução.

In [ ]:
# =============================================================================
# MÉTRICAS DE AVALIAÇÃO POR TAREFA
# -----------------------------------------------------------------------------
# Cada métrica vem com a sua DIREÇÃO: se o melhor valor é o MAIOR, o MENOR ou o
# mais PRÓXIMO DE ZERO. O AutoML (notebook 03) usa a direção para ordenar as
# técnicas e escolher a vencedora; a tabela comparativa usa a direção para dizer
# se a variação teste → OOT é melhora ou piora.
#
# PD (alvo 0/1 — o contrato inadimpliu ou não em 12 meses):
#   ks         Kolmogorov-Smirnov: maior distância entre as distribuições de score
#              dos bons e dos maus. Mede ORDENAÇÃO (separação). Faixa usual em
#              crédito: 0,20 a 0,50; acima de 0,75 costuma indicar vazamento.
#   auc        área sob a curva ROC: chance de um mau sorteado ao acaso ter PD
#              maior que um bom sorteado ao acaso. 0,5 = acaso; 1,0 = perfeito.
#   gini       2·AUC − 1. A mesma informação da AUC, na escala usual de crédito.
#   log_loss   entropia cruzada: penaliza probabilidades CONFIANTES e ERRADAS.
#              Mede calibração e ordenação ao mesmo tempo. Menor é melhor.
#   brier      erro quadrático médio entre a PD prevista e o alvo 0/1. Também
#              mede calibração e ordenação. Menor é melhor.
#   acuracia, precisao, revocacao
#              calculadas depois de cortar o score no ponto KS-ótimo do TESTE
#              (o corte que maximiza o KS). Dependem do corte escolhido:
#              acurácia = acertos / total; precisão = maus corretos / previstos
#              como maus; revocação = maus corretos / maus reais.
#
# LGD (alvo contínuo entre 0 e 1 — fração da exposição perdida):
#   rmse       raiz do erro quadrático médio. Penaliza erros grandes.
#   mae        erro absoluto médio. Mais robusto a valores extremos.
#   r2         fração da variação da LGD explicada pelo modelo.
#   vies       média de (previsto − realizado). Positivo = conservador
#              (superestima a perda). O melhor é o mais perto de zero.
#   spearman   correlação de Spearman (correlação entre as ORDENS) entre previsto
#              e realizado. É o análogo do KS para LGD: mede só a ordenação.
# =============================================================================
METRICAS_POR_TAREFA = {
    "pd": {
        "ks": "maior",
        "auc": "maior",
        "gini": "maior",
        "log_loss": "menor",
        "brier": "menor",
        "acuracia": "maior",
        "precisao": "maior",
        "revocacao": "maior",
    },
    "lgd": {
        "rmse": "menor",
        "mae": "menor",
        "r2": "maior",
        "vies": "proximo_de_zero",
        "spearman": "maior",
    },
}

# =============================================================================
# TÉCNICAS DE APRENDIZADO DE MÁQUINA POR TAREFA (todas treinadas em Spark)
# -----------------------------------------------------------------------------
# PD:
#   logistica       regressão logística (pyspark.ml). Soma ponderada das variáveis
#                   na escala do logit, ln(PD / (1 − PD)). É a referência linear:
#                   simples de auditar, mas só aprende relações "em linha reta".
#   random_forest   floresta aleatória (pyspark.ml): média de centenas de árvores,
#                   cada uma treinada numa amostra sorteada da base.
#   gbt             gradient boosted trees (pyspark.ml): árvores em sequência,
#                   cada uma corrigindo o erro das anteriores.
#   lightgbm        boosting rápido, árvores que crescem folha a folha (SynapseML).
#   xgboost         boosting com regularização (xgboost.spark).
#
# LGD:
#   logistica_fracionaria
#                   GeneralizedLinearRegression com família binomial e alvo entre
#                   0 e 1: o análogo da logística para LGD. Só prevê dentro de [0, 1].
#   random_forest, gbt, lightgbm, xgboost
#                   versões de regressão. LightGBM e XGBoost usam objetivos que
#                   só preveem dentro de [0, 1] (ver PADRAO_REFERENCIA).
#   duas_etapas     modelo "hurdle", feito para a LGD bimodal (concentrada em 0 e 1):
#                   etapa 1 estima P(LGD ≥ limiar); etapa 2 estima a LGD só entre os
#                   contratos com perda; resposta = P · LGD_etapa2 + (1 − P) · média
#                   das curas.
# =============================================================================
TECNICAS_POR_TAREFA = {
    "pd": ["logistica", "random_forest", "gbt", "lightgbm", "xgboost"],
    "lgd": ["logistica_fracionaria", "random_forest", "gbt", "lightgbm", "xgboost", "duas_etapas"],
}

# =============================================================================
# PADRÃO DE REFERÊNCIA (hiperparâmetros da comparação entre técnicas)
# -----------------------------------------------------------------------------
# Decisão de desenho: cada técnica é treinada UMA vez com este padrão, a vencedora
# no OOT é otimizada depois. O padrão de fábrica das bibliotecas Spark é desigual
# (Random Forest e GBT vêm com só 20 árvores rasas), o que faria a comparação
# medir a configuração, e não a técnica. Aqui todas as técnicas de árvore têm
# 300 árvores (ou iterações) e, nos boostings, a mesma taxa de aprendizado (0,05:
# o tamanho de cada correção que uma árvore nova faz sobre as anteriores).
#
# Os nomes dos parâmetros são os da API de cada biblioteca (por isso em inglês):
#   pyspark.ml   → regParam, numTrees, maxDepth, maxIter, stepSize...
#   SynapseML    → numIterations, numLeaves, learningRate, objective
#   xgboost.spark→ n_estimators, max_depth, learning_rate, objective
#
# Objetivos limitados a [0, 1] na LGD (decisão de desenho): com erro quadrático
# um boosting pode prever LGD de −5% ou 112%. "cross_entropy" (LightGBM) e
# "reg:logistic" (XGBoost) aceitam alvo contínuo entre 0 e 1 e só preveem nesse
# intervalo.
# =============================================================================
PADRAO_REFERENCIA = {
    "pd": {
        "logistica": {"regParam": 0.01, "elasticNetParam": 0.0, "maxIter": 100, "standardization": True},
        "random_forest": {"numTrees": 300, "maxDepth": 8, "featureSubsetStrategy": "sqrt"},
        "gbt": {"maxIter": 300, "maxDepth": 5, "stepSize": 0.05},
        "lightgbm": {"numIterations": 300, "numLeaves": 31, "learningRate": 0.05, "objective": "binary"},
        # O classificador do xgboost.spark define o objetivo sozinho (binary:logistic)
        # e recusa o parâmetro "objective".
        "xgboost": {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05},
    },
    "lgd": {
        "logistica_fracionaria": {"family": "binomial", "link": "logit", "regParam": 0.01, "maxIter": 100},
        "random_forest": {"numTrees": 300, "maxDepth": 8, "featureSubsetStrategy": "sqrt"},
        "gbt": {"maxIter": 300, "maxDepth": 5, "stepSize": 0.05},
        "lightgbm": {"numIterations": 300, "numLeaves": 31, "learningRate": 0.05, "objective": "cross_entropy"},
        "xgboost": {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05, "objective": "reg:logistic"},
        # Duas etapas: as duas partes usam LightGBM com o padrão acima
        # (etapa 1 com objetivo "binary", etapa 2 com "cross_entropy").
        # Ajustes feitos pelo usuário na técnica de cada etapa (ex.: numIterations
        # do "lightgbm") valem também para a etapa; o objetivo de cada etapa é fixo.
        # O limiar que separa "cura" de "perda" fica em cfg["automl"]["limiar_duas_etapas"].
        "duas_etapas": {"tecnica_etapa1": "lightgbm", "tecnica_etapa2": "lightgbm"},
    },
}

# Métodos de calibração aceitos por tarefa (etapa 5):
#   platt      regressão logística sobre o logit do score: corrige nível e inclinação.
#   isotonica  ajuste em degraus que só sobe: corrige qualquer forma monotônica.
#   linear     a·previsto + b, com o resultado limitado a [0, 1] (só LGD).
METODOS_CALIBRACAO = {"pd": ["platt", "isotonica"], "lgd": ["linear", "isotonica"]}

## 2. Configuração padrão

Cada valor abaixo é uma **decisão do desenho**. O comentário ao lado diz em que etapa o parâmetro é usado e por quê. Qualquer valor pode ser alterado em `cria_configuracao(..., ajustes={...})`, sem editar este notebook.

In [ ]:
# Nomes das colunas de entrada (contrato de dados, só colunas essenciais).
COLUNAS_PADRAO = {
    "pd": {
        "id": "id_contrato",                                   # identificador único do contrato
        "safra": "safra",                                      # mês de concessão (1º dia do mês)
        "amostra": "amostra",                                  # DES, OOT ou HIST
        "alvo": "alvo_12m",                                    # 1 = default até 12 meses após a concessão
        # As duas datas abaixo são OPCIONAIS. Só a PD lifetime (etapa 8) precisa
        # delas. Se a base não as tiver, ajuste as duas para None: as etapas 1 a 7
        # rodam normalmente e a etapa 8 é pulada.
        "data_primeiro_default": "data_primeiro_default",      # 1º mês em default (vazio = nunca)
        "data_fim_observacao": "data_fim_observacao",          # último mês acompanhado
    },
    "lgd": {
        "id": "id_contrato",
        "safra": "safra_default",                              # mês do default (faz o papel da safra)
        "amostra": "amostra",                                  # DES ou OOT
        "alvo": "lgd_realizada",                               # perda final / exposição no default, em [0, 1]
    },
}

# Rótulos aceitos na coluna de amostra. DES e OOT são obrigatórios. O HIST é
# opcional e vem desligado (None): só a PD lifetime o usa. Para ligá-lo, informe
# o rótulo em ajustes, ex.: {"rotulos_amostra": {"hist": "HIST"}}.
ROTULOS_AMOSTRA = {"des": "DES", "oot": "OOT", "hist": None}


def _configuracao_padrao(tarefa):
    """Devolve o dicionário de configuração com os valores do desenho para a tarefa."""
    return {
        # ------------------------------------------------------------------ geral
        "tarefa": tarefa,                       # "pd" ou "lgd"
        "metrica": None,                        # métrica que escolhe a técnica vencedora e guia o Optuna
        "tecnicas": list(TECNICAS_POR_TAREFA[tarefa]),
        "semente": 42,                          # semente única de TODOS os sorteios (reprodutibilidade)
        "data_corte": None,                     # último mês com histórico disponível, "AAAA-MM-01"
        "esquema_saida": None,                  # "catalogo.esquema" do Unity Catalog para as tabelas Delta
        "prefixo_tabelas": "desafiante",        # tabelas: <esquema>.<prefixo>_<tarefa>_<nome>
        "experimento_mlflow": None,             # caminho do experimento no MLflow
        # Pasta (Volume do Unity Catalog ou DBFS) onde os modelos Spark ajustados são
        # salvos, em <caminho>/<id_execucao>/. None = não salva (só fica na sessão).
        "caminho_artefatos": None,
        # Esquema ("catalogo.esquema") para as tabelas temporárias que substituem o
        # persist no serverless (ver seção 4b). None = usa o esquema_saida.
        "esquema_temporario": None,
        # Bases largas (milhares de variáveis): as agregações por variável rodam em
        # lotes deste tamanho, para manter cada plano Spark pequeno.
        "tamanho_lote_variaveis": 500,
        "colunas": copy.deepcopy(COLUNAS_PADRAO[tarefa]),
        "variaveis": None,                      # lista de variáveis explicativas; None = todas as não-controle
        "rotulos_amostra": dict(ROTULOS_AMOSTRA),

        # ------------------------------------------------------------- etapa 1
        "entrada": {
            # PD: uma safra só tem alvo de 12 meses válido se já passaram 12 meses
            # até a data de corte. Antes disso, parte dos maus ainda não apareceu.
            "janela_desempenho_meses": 12,
            # LGD: só entram safras de default com pelo menos 24 meses até o corte.
            # Safras recentes têm poucas cobranças encerradas, e as que encerraram
            # cedo são curas rápidas ou baixas rápidas: a LGD média sai distorcida.
            "janela_cobranca_meses": 24,
        },
        # Divisão do DES pela ordem das safras: as primeiras ~80% do volume viram
        # treino, as últimas viram teste. Uma safra nunca é partida.
        "divisao": {"proporcao_treino": 0.80},

        # --------------------------------------------- pré-processamento (03/04)
        "preprocessamento": {
            # Categorias com menos de 1% dos contratos viram "OUTROS" (evita colunas
            # quase vazias e instáveis). Categoria nova no OOT também vira "OUTROS".
            "piso_categoria_rara": 0.01,
            "rotulo_categoria_rara": "OUTROS",
            # Técnicas que NÃO aceitam valores vazios: recebem mediana do treino +
            # variável indicadora "<variavel>_faltante". LightGBM e XGBoost recebem
            # os dados como estão (cada árvore aprende para onde mandar o vazio).
            "tecnicas_com_imputacao": ["logistica", "logistica_fracionaria", "random_forest", "gbt"],
            "sufixo_indicador_faltante": "_faltante",
        },

        # ------------------------------------------------------------- etapa 2
        "prefiltros": {
            "max_faltantes": 0.95,                       # descarta variável com > 95% vazia
            "max_participacao_valor_dominante": 0.99,    # descarta quase constante (um valor em > 99%)
            "max_psi": 0.25,                             # descarta instável entre DES e OOT
            "min_iv": 0.02,                              # PD: descarta IV < 0,02 (sem poder)
            "min_spearman_alvo": 0.05,                   # LGD: descarta |Spearman| < 0,05
            "max_correlacao": 0.85,                      # de cada par acima disso, fica a mais forte
            "n_faixas": 10,                              # faixas (quantis do treino) para PSI e IV
            # Funil para bases largas: as etapas caras recebem menos variáveis e menos linhas.
            "max_variaveis_correlacao": 1500,            # teto 1: só as N mais fortes seguem para a correlação
            "n_linhas_correlacao": 20_000,               # Spearman numa amostra do treino (erro ≈ 0,002 em ρ = 0,85)
            "max_variaveis_boruta": 400,                 # teto 2: só as M mais fortes (após redundância) vão ao Boruta
            "forca_alerta_teto": 0.1,                    # avisa se um teto cortar variável com IV (ou η) ≥ 0,1
            # Aviso de vazamento: poder bom demais sugere informação do futuro.
            # Só marca; a decisão de descartar é de quem analisa.
            "vazamento_iv": 0.5,
            "vazamento_ks": 0.5,
            "vazamento_spearman": 0.7,
        },

        # ------------------------------------------------------------- etapa 3
        "boruta": {
            "n_rodadas": 50,                    # rodadas de LightGBM com variáveis-sombra
            "nivel_significancia": 0.01,        # teste binomial a 1% (99% de confiança), com Bonferroni
            # Amostra da PD: "caso_controle" (padrão) = todos os maus (até max_maus) e
            # razao_bons_por_mau bons sorteados para cada mau. O Boruta compara
            # importâncias; quem dá informação a essa comparação são os maus, então
            # mantê-los todos preserva a qualidade com muito menos linhas.
            # "formula" = tamanho pela fórmula de confiança abaixo (com evento raro,
            # quase sempre resulta no treino inteiro).
            "amostragem_pd": "caso_controle",
            "razao_bons_por_mau": 9,            # ex.: 1.413 maus → ~12.700 bons → ~14.100 linhas
            "max_maus": 50_000,                 # acima disso, os maus também são sorteados
            "confianca_amostra": 0.99,          # fórmula (LGD; PD com "formula"): 99% de confiança...
            "margem_relativa": 0.02,            # ...e margem de 2% da taxa (PD) ou da média (LGD)
            "limite_treino_inteiro": 300_000,   # LGD: abaixo disso usa o treino inteiro, sem amostrar
            # Medida de importância comparada com as sombras: "shap" (média do |SHAP|,
            # calculada pelo próprio LightGBM) ou "ganho". Os testes do pipeline mostraram
            # que o ganho favorece variáveis contínuas: uma categórica informativa (IV 0,12)
            # perdeu para sombras de ruído em 100% das rodadas. Com SHAP e o LightGBM
            # regularizado abaixo, venceu em 100%, e o ruído não venceu nenhuma.
            "medida_importancia": "shap",
            "n_linhas_shap": 5_000,             # linhas sorteadas por rodada para o SHAP (a média do |SHAP| já é estável)
            # Folha mínima como fração da amostra (vira min_child_samples e
            # min_data_per_group): impede o modelo de criar folhas minúsculas que
            # ajustam ruído, que é o que dá importância às sombras.
            "participacao_minima_folha": 0.01,
            "hiperparametros_lightgbm": {
                # 150 árvores com taxa 0,1: ajuste equivalente ao de 300 com 0,05, na metade do tempo.
                "n_estimators": 150, "learning_rate": 0.1, "num_leaves": 15,
                "subsample": 0.8, "subsample_freq": 1,      # cada árvore vê 80% dos contratos
                "colsample_bytree": 0.8,                     # e 80% das variáveis
                "cat_smooth": 50,                            # suaviza a taxa de categorias pequenas
            },
        },

        # ------------------------------------------------------------- etapa 4
        "automl": {
            "padrao_referencia": copy.deepcopy(PADRAO_REFERENCIA[tarefa]),
            "usar_dois_finalistas": False,      # se True e a diferença for < margem, otimiza os dois primeiros
            "margem_empate": 0.01,              # na escala da métrica escolhida
            "n_tentativas_optuna": 50,          # tentativas da otimização do vencedor (0 = não otimiza)
            "tempo_maximo_optuna_min": None,    # limite de tempo opcional (minutos)
            "limiar_duas_etapas": 0.05,         # LGD: separa "cura" (LGD < 5%) de "perda"
        },

        # ------------------------------------------------------------- etapa 5
        "calibracao": {"metodo": "platt" if tarefa == "pd" else "linear"},

        # ------------------------------------------------------------- etapa 6
        "explicabilidade": {
            "tamanho_amostra": 50_000,          # amostra para o espelho em pandas e o SHAP
            "n_pontos_beeswarm": 5_000,         # contratos desenhados no beeswarm (legibilidade)
            "n_variaveis_grafico": 20,          # variáveis no beeswarm e no gráfico de importância
            "fidelidade_min_spearman": 0.95,    # score Spark × score do espelho
            "fidelidade_min_mesmo_rating": 0.90,
            "fidelidade_top_k": 10,             # sobreposição das k variáveis de maior ganho
        },

        # ------------------------------------------------------------- etapa 7
        "regua": {
            "n_faixas": 10,                     # faixas iniciais (quantis); ou "auto"
            "n_faixas_min": 3,                  # limites do modo "auto"
            "n_faixas_max": 15,
            "nivel_fusao": 0.05,                # nível dos testes entre faixas vizinhas
            "fundir_indistinguiveis": False,    # funde também vizinhos em ordem mas sem diferença estatística
            "piso_representatividade": 0.02,    # rating com < 2% dos contratos é fundido ao vizinho
        },

        # ------------------------------------------------------------ etapa 7b
        "backtest_regua": {
            "nivel": 0.95,                      # intervalo de confiança da taxa (ou média) observada
            "tol_relativa": 0.10,               # materialidade: |observada / régua − 1| ≥ 10%
            "n_safras_persistencia": 3,         # alertas seguidos para marcar "recalibrar"
        },

        # ------------------------------------------------------------- etapa 8
        "lifetime": None if tarefa == "lgd" else {
            "metodo": "vintage",                # costura de PDs condicionais por idade
            "horizonte_meses": 60,              # até onde a curva é projetada
            "horizontes_backtest": [12, 24, 36],
            # Idade com menos contratos em risco que isso não estima o hazard: a
            # taxa seria ruído. Essas idades usam o hazard médio das últimas idades
            # confiáveis (extrapolação plana).
            "min_em_risco": 100,
            "n_idades_extrapolacao": 6,
        },
    }

## 3. Criação e validação da configuração

`cria_configuracao` parte do padrão da tarefa, aplica os ajustes do usuário e valida tudo. Um ajuste com nome errado (por exemplo `"boruta": {"n_rodada": 30}`) gera erro, em vez de ser ignorado em silêncio.

In [ ]:
def mescla_dicionarios(base, ajustes, caminho=""):
    """
    Aplica `ajustes` sobre `base`, recursivamente, e devolve uma cópia nova.

    Conceito técnico:
        Só é permitido ajustar chaves que já existem no padrão. Uma chave
        desconhecida quase sempre é erro de digitação, e ignorá-la faria a
        execução rodar com o valor padrão sem ninguém perceber.
    """
    resultado = copy.deepcopy(base)
    for chave, valor in (ajustes or {}).items():
        caminho_chave = f"{caminho}.{chave}" if caminho else chave
        if chave not in resultado:
            raise KeyError(f"Parâmetro desconhecido na configuração: '{caminho_chave}'.")
        if isinstance(resultado[chave], dict) and isinstance(valor, dict):
            resultado[chave] = mescla_dicionarios(resultado[chave], valor, caminho_chave)
        else:
            resultado[chave] = copy.deepcopy(valor)
    return resultado


def valida_configuracao(cfg):
    """
    Confere a consistência da configuração e levanta erro listando TODOS os
    problemas encontrados (e não só o primeiro).
    """
    problemas = []
    tarefa = cfg.get("tarefa")
    if tarefa not in METRICAS_POR_TAREFA:
        raise ValueError(f"tarefa deve ser 'pd' ou 'lgd'; recebido {tarefa!r}.")

    # Métrica e técnicas precisam existir para a tarefa.
    if cfg["metrica"] not in METRICAS_POR_TAREFA[tarefa]:
        problemas.append(f"metrica {cfg['metrica']!r} inválida para {tarefa}; "
                         f"opções: {sorted(METRICAS_POR_TAREFA[tarefa])}")
    if not cfg["tecnicas"]:
        problemas.append("a lista de técnicas está vazia")
    desconhecidas = [t for t in cfg["tecnicas"] if t not in TECNICAS_POR_TAREFA[tarefa]]
    if desconhecidas:
        problemas.append(f"técnicas desconhecidas para {tarefa}: {desconhecidas}")
    sem_padrao = [t for t in cfg["tecnicas"] if t not in cfg["automl"]["padrao_referencia"]]
    if sem_padrao:
        problemas.append(f"técnicas sem padrão de referência: {sem_padrao}")

    # A data de corte precisa ser o 1º dia de um mês: todas as datas do
    # pipeline são mensais e as contas de idade usam meses-calendário.
    try:
        corte = date.fromisoformat(str(cfg["data_corte"]))
        if corte.day != 1:
            problemas.append("data_corte deve ser o 1º dia do mês (AAAA-MM-01)")
    except (TypeError, ValueError):
        problemas.append(f"data_corte inválida: {cfg['data_corte']!r} (use 'AAAA-MM-01')")

    # Proporções e níveis de confiança precisam estar entre 0 e 1.
    verificacoes_unitarias = {
        "divisao.proporcao_treino": cfg["divisao"]["proporcao_treino"],
        "boruta.nivel_significancia": cfg["boruta"]["nivel_significancia"],
        "boruta.confianca_amostra": cfg["boruta"]["confianca_amostra"],
        "boruta.margem_relativa": cfg["boruta"]["margem_relativa"],
        "boruta.participacao_minima_folha": cfg["boruta"]["participacao_minima_folha"],
        "regua.nivel_fusao": cfg["regua"]["nivel_fusao"],
        "regua.piso_representatividade": cfg["regua"]["piso_representatividade"],
        "backtest_regua.nivel": cfg["backtest_regua"]["nivel"],
        "preprocessamento.piso_categoria_rara": cfg["preprocessamento"]["piso_categoria_rara"],
    }
    for nome, valor in verificacoes_unitarias.items():
        if not (isinstance(valor, (int, float)) and 0 < valor < 1):
            problemas.append(f"{nome} deve estar entre 0 e 1; recebido {valor!r}")

    # Número de faixas da régua: inteiro dentro dos limites, ou "auto".
    n_faixas = cfg["regua"]["n_faixas"]
    if n_faixas != "auto":
        if not isinstance(n_faixas, int) or not (
                cfg["regua"]["n_faixas_min"] <= n_faixas <= cfg["regua"]["n_faixas_max"]):
            problemas.append(f"regua.n_faixas deve ser 'auto' ou inteiro entre "
                             f"{cfg['regua']['n_faixas_min']} e {cfg['regua']['n_faixas_max']}")

    if cfg["boruta"]["amostragem_pd"] not in ("caso_controle", "formula"):
        problemas.append("boruta.amostragem_pd deve ser 'caso_controle' ou 'formula'")
    if cfg["boruta"]["medida_importancia"] not in ("shap", "ganho"):
        problemas.append("boruta.medida_importancia deve ser 'shap' ou 'ganho'")

    if cfg["calibracao"]["metodo"] not in METODOS_CALIBRACAO[tarefa]:
        problemas.append(f"calibracao.metodo deve ser um de {METODOS_CALIBRACAO[tarefa]}")

    # As datas de evento da PD andam juntas: ou as duas, ou nenhuma.
    if tarefa == "pd":
        informadas = [cfg["colunas"]["data_primeiro_default"] is not None,
                      cfg["colunas"]["data_fim_observacao"] is not None]
        if any(informadas) and not all(informadas):
            problemas.append("colunas.data_primeiro_default e colunas.data_fim_observacao devem ser "
                             "informadas juntas (ou as duas None, sem PD lifetime)")

    if tarefa == "lgd" and "duas_etapas" in cfg["tecnicas"]:
        limiar = cfg["automl"]["limiar_duas_etapas"]
        if not (0 < limiar < 1):
            problemas.append("automl.limiar_duas_etapas deve estar entre 0 e 1")

    if problemas:
        raise ValueError("Configuração inválida:\n  - " + "\n  - ".join(problemas))
    return True


def tem_datas_de_evento(cfg):
    """
    True quando a base de PD traz as datas de default e de fim de observação.
    Sem elas, a validação dessas datas e a PD lifetime (etapa 8) são puladas.
    """
    return (cfg["tarefa"] == "pd"
            and cfg["colunas"].get("data_primeiro_default") is not None
            and cfg["colunas"].get("data_fim_observacao") is not None)


def cria_configuracao(tarefa, metrica, data_corte, esquema_saida=None,
                      experimento_mlflow=None, ajustes=None):
    """
    Monta a configuração de uma execução do pipeline.

    Parâmetros
        tarefa              "pd" ou "lgd".
        metrica             métrica que escolhe a técnica vencedora e guia o Optuna
                            (ver METRICAS_POR_TAREFA).
        data_corte          último mês com histórico disponível, "AAAA-MM-01".
                            Define a maturidade das safras e a censura da PD lifetime.
        esquema_saida       "catalogo.esquema" onde as tabelas Delta são gravadas.
        experimento_mlflow  caminho do experimento no MLflow.
        ajustes             dicionário com os parâmetros a alterar em relação ao
                            padrão, na mesma estrutura do padrão. Ex.:
                            {"regua": {"n_faixas": "auto"}, "boruta": {"n_rodadas": 30}}

    Devolve
        Dicionário de configuração, já validado, com um `id_execucao` único que
        acompanha todas as tabelas e registros desta execução.
    """
    tarefa = str(tarefa).lower()
    if tarefa not in METRICAS_POR_TAREFA:
        raise ValueError(f"tarefa deve ser 'pd' ou 'lgd'; recebido {tarefa!r}.")
    cfg = mescla_dicionarios(_configuracao_padrao(tarefa), ajustes)
    cfg["metrica"] = metrica
    cfg["data_corte"] = str(data_corte)
    cfg["esquema_saida"] = esquema_saida
    cfg["experimento_mlflow"] = experimento_mlflow
    # id_execucao: data e hora + 6 caracteres aleatórios. Vai em toda tabela
    # gravada, para separar e comparar execuções.
    agora = datetime.now()
    cfg["id_execucao"] = f"{agora:%Y%m%d_%H%M%S}_{uuid.uuid4().hex[:6]}"
    cfg["data_execucao"] = agora.isoformat(timespec="seconds")
    valida_configuracao(cfg)
    return cfg


def mostra_configuracao(cfg):
    """Imprime a configuração em JSON legível."""
    print(json.dumps(cfg, indent=2, ensure_ascii=False, default=str))

## 4. Utilitários de saída

Funções usadas por todas as etapas para exibir tabelas, gravar resultados em Delta e registrar a execução no MLflow.

**Por que Delta e MLflow.** O MLflow guarda gráficos, modelos e a configuração de cada execução. As tabelas de resultado (variáveis, comparação de técnicas, régua, backtest, curvas) vão para tabelas Delta com a coluna `id_execucao`, porque assim podem ser consultadas e comparadas entre execuções com SQL.

In [ ]:
def exibe(tabela, titulo=None):
    """
    Mostra uma tabela pandas: com `display` no Databricks/Jupyter, com texto fora deles.
    """
    if titulo:
        print(f"\n=== {titulo} ===")
    try:
        display(tabela)  # noqa: F821 — função global do Databricks e do Jupyter
    except NameError:
        print(tabela.to_string(index=False))


def achata_dicionario(dicionario, prefixo=""):
    """
    Transforma {"boruta": {"n_rodadas": 50}} em {"boruta.n_rodadas": 50}.
    O MLflow guarda parâmetros como pares chave–valor simples.
    """
    plano = {}
    for chave, valor in dicionario.items():
        nome = f"{prefixo}.{chave}" if prefixo else str(chave)
        if isinstance(valor, dict):
            plano.update(achata_dicionario(valor, nome))
        else:
            plano[nome] = valor
    return plano


def registra_versoes_bibliotecas():
    """
    Versões das bibliotecas que afetam o resultado (reprodutibilidade).

    Observação: o SynapseML instalado como biblioteca Maven do cluster não aparece
    no registro de pacotes do Python; nesse caso a versão vem como "não encontrada"
    e deve ser conferida na aba Libraries do cluster.
    """
    pacotes = ["pyspark", "synapseml", "lightgbm", "xgboost", "shap", "optuna",
               "mlflow", "pandas", "numpy", "scipy", "matplotlib"]
    versoes = {}
    for pacote in pacotes:
        try:
            versoes[pacote] = metadata.version(pacote)
        except metadata.PackageNotFoundError:
            versoes[pacote] = "não encontrada"
    return versoes


def nome_tabela_saida(cfg, nome):
    """Nome completo da tabela Delta: <esquema>.<prefixo>_<tarefa>_<nome>."""
    return f"{cfg['esquema_saida']}.{cfg['prefixo_tabelas']}_{cfg['tarefa']}_{nome}"


def grava_tabela_delta(spark, tabela, nome, cfg, modo="append"):
    """
    Grava uma tabela de resultado em Delta, acrescentando `id_execucao` e
    `data_execucao`.

    Parâmetros
        tabela  DataFrame pandas (tabelas pequenas de resultado) ou Spark.
        nome    sufixo da tabela, ex.: "entrada_validacao".
        modo    "append" (padrão): cada execução acrescenta linhas, separadas
                pelo id_execucao; o histórico de execuções fica na mesma tabela.

    Conceito técnico:
        mergeSchema permite que execuções futuras acrescentem colunas novas sem
        quebrar a tabela.
    """
    from pyspark.sql import functions as F
    if not cfg.get("esquema_saida"):
        print(f"[aviso] esquema_saida não definido: tabela '{nome}' não foi gravada.")
        return None
    if not hasattr(tabela, "write"):                 # pandas → Spark
        tabela = spark.createDataFrame(tabela)
    tabela = (tabela
              .withColumn("id_execucao", F.lit(cfg["id_execucao"]))
              .withColumn("data_execucao", F.lit(cfg["data_execucao"])))
    nome_completo = nome_tabela_saida(cfg, nome)
    (tabela.write.format("delta").mode(modo)
           .option("mergeSchema", "true").saveAsTable(nome_completo))
    return nome_completo


def registra_figura_mlflow(figura, caminho):
    """
    Registra um gráfico (figura matplotlib) na execução MLflow ativa, se houver.
    Sem execução ativa (ou sem MLflow), não faz nada: o gráfico continua
    disponível no notebook.

    `caminho` é o nome do arquivo dentro da execução, ex.: "selecao/boruta.png".
    """
    try:
        import mlflow
        if mlflow.active_run() is not None:
            mlflow.log_figure(figura, caminho)
    except ImportError:
        pass


def registra_configuracao_mlflow(cfg):
    """
    Registra a configuração e as versões das bibliotecas na execução MLflow ativa.

    Os parâmetros vão achatados (para filtro e comparação na interface do MLflow)
    e também como JSON completo (para reproduzir a execução).
    """
    import mlflow
    parametros = {chave: str(valor)[:500] for chave, valor in achata_dicionario(cfg).items()}
    mlflow.log_params(parametros)
    mlflow.log_dict(cfg, "configuracao.json")
    mlflow.log_dict(registra_versoes_bibliotecas(), "versoes_bibliotecas.json")

## 4b. Resultados intermediários: persist ou tabela temporária

Algumas bases intermediárias são lidas muitas vezes (o treino, por exemplo, é relido por cada técnica e por cada tentativa do Optuna). Sem guardar o resultado, o Spark refaz todo o cálculo desde a tabela de entrada a cada leitura.

| ambiente | como o resultado é guardado |
|---|---|
| cluster clássico | `persist()`: o Spark guarda o resultado na memória e no disco dos nós |
| **serverless** | `persist()` não é permitido (gera erro). O resultado é gravado numa **tabela temporária** em `esquema_temporario` (ou `esquema_saida`), chamada `<prefixo>_tmp_<id_execucao>_<nome>`, e lido de volta. As tabelas temporárias são apagadas no fim do orquestrador (`apaga_tabelas_temporarias`). |
| serverless sem esquema definido | o resultado não é guardado: funciona, mas cada leitura refaz o cálculo (mais lento) |

A escolha é automática: `guarda_resultado` tenta o `persist()` e, se o ambiente não permitir, usa a tabela temporária.

In [ ]:
_ESTADO_PERSIST = {"disponivel": None}   # None = ainda não testado na sessão
_TABELAS_TEMPORARIAS = []                # tabelas temporárias criadas nesta sessão


def persiste_se_possivel(sdf):
    """
    Tenta sdf.persist(); no serverless (onde persist gera erro) devolve o
    próprio DataFrame, sem guardar. Para conjuntos pequenos ou de uso curto.
    """
    if _ESTADO_PERSIST["disponivel"] is False:
        return sdf
    try:
        resultado = sdf.persist()
        _ESTADO_PERSIST["disponivel"] = True
        return resultado
    except Exception:
        if _ESTADO_PERSIST["disponivel"] is None:
            print("[aviso] persist() não é permitido neste ambiente (serverless): "
                  "os resultados intermediários usam tabelas temporárias quando houver esquema definido.")
        _ESTADO_PERSIST["disponivel"] = False
        return sdf


def guarda_resultado(spark, sdf, nome, cfg):
    """
    Guarda um resultado intermediário lido muitas vezes.

    1. Tenta persist() (cluster clássico).
    2. Se não for permitido (serverless), grava em tabela temporária no
       esquema_temporario (ou esquema_saida) e devolve a leitura dela.
    3. Sem esquema definido, devolve o próprio DataFrame (recalculado a cada leitura).
    """
    resultado = persiste_se_possivel(sdf)
    if _ESTADO_PERSIST["disponivel"]:
        return resultado
    esquema = cfg.get("esquema_temporario") or cfg.get("esquema_saida")
    if not esquema:
        if not _ESTADO_PERSIST.get("avisou_sem_esquema"):
            print("[aviso] serverless sem esquema_temporario/esquema_saida: os resultados intermediários "
                  "não são guardados e cada leitura refaz o cálculo desde a base de entrada (lento em "
                  "bases largas). Defina \"esquema_temporario\" no AJUSTES.")
            _ESTADO_PERSIST["avisou_sem_esquema"] = True
        return sdf
    nome_tabela = f"{esquema}.{cfg['prefixo_tabelas']}_tmp_{cfg['id_execucao']}_{nome}"
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(nome_tabela)
    if nome_tabela not in _TABELAS_TEMPORARIAS:
        _TABELAS_TEMPORARIAS.append(nome_tabela)
    return spark.table(nome_tabela)


def em_lotes(lista, tamanho):
    """Divide `lista` em pedaços de até `tamanho` elementos (agregações em lotes de variáveis)."""
    lista = list(lista)
    return [lista[i:i + tamanho] for i in range(0, len(lista), max(1, int(tamanho)))]


def libera_resultado(sdf):
    """Libera um resultado guardado com persist (no serverless, não faz nada)."""
    if _ESTADO_PERSIST["disponivel"]:
        try:
            sdf.unpersist()
        except Exception:
            pass


def apaga_tabelas_temporarias(spark):
    """Apaga as tabelas temporárias criadas nesta sessão (chamada no fim do orquestrador)."""
    while _TABELAS_TEMPORARIAS:
        nome_tabela = _TABELAS_TEMPORARIAS.pop()
        try:
            spark.sql(f"DROP TABLE IF EXISTS {nome_tabela}")
        except Exception as erro:
            print(f"[aviso] não foi possível apagar {nome_tabela}: {erro}")

## 5. Exemplo de uso

```python
cfg_pd = cria_configuracao(
    tarefa="pd", metrica="ks", data_corte="2025-09-01",
    esquema_saida="catalogo.risco_desafiante",
    experimento_mlflow="/Users/voce@banco.com/desafiante_pd",
    ajustes={"regua": {"n_faixas": "auto"}},
)
mostra_configuracao(cfg_pd)

cfg_lgd = cria_configuracao(tarefa="lgd", metrica="rmse", data_corte="2025-09-01",
                            esquema_saida="catalogo.risco_desafiante")
```